# 5b. Final Multi-Model Pipeline & Optimization

**Goal:** In previous notebooks, we determined that Crohn's Disease (CD) and Ulcerative Colitis (UC) possess different microbial signatures. 

In this notebook, we finalize our machine learning pipelines. Based on our exploratory analysis, we establish **fixed, strict feature selection thresholds**:
1. **Kruskal-Wallis (KW):** Strict raw $p$-value $< 0.01$.
2. **Principal Component Analysis (PCA):** Extract components explaining exactly $60\%$ of the variance.

**Objective:**
We will test **8 distinct configurations** (Task $\times$ Feature Selection $\times$ Classifier) to systematically compare non-linear models (SVM) against linear baselines (Logistic Regression). 

The 8 models are:
1. UC: KW + SVM
2. UC: KW + LogReg
3. UC: PCA + SVM
4. UC: PCA + LogReg
5. CD: KW + SVM
6. CD: KW + LogReg
7. CD: PCA + SVM
8. CD: PCA + LogReg

Each pipeline is tuned to prevent data leakage and rigorously evaluated via a 3-run (15-fold) cross-validation to report true expected performance.

In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import joblib
import os
from scipy.stats import kruskal

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings("ignore")

# Ensure results directories exist
os.makedirs('../results', exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# 2. Load Data and Prepare Binary Tasks
meta = pd.read_csv("../data/processed/metadata_final.tsv", sep="\t", index_col="Sample")
genera_clr = pd.read_csv("../data/processed/genera_clr.tsv", sep="\t", index_col=0)
meta = meta.loc[genera_clr.index]

def make_binary_dataset(meta, genera, group_a, group_b):
    mask = meta["Study.Group"].isin([group_a, group_b])
    meta_sub = meta[mask]
    X = genera.loc[meta_sub.index]
    y = (meta_sub["Study.Group"] == group_a).astype(int)
    return X, y

X_uc, y_uc = make_binary_dataset(meta, genera_clr, "UC", "nonIBD")
X_cd, y_cd = make_binary_dataset(meta, genera_clr, "CD", "nonIBD")

print(f"UC vs nonIBD: {X_uc.shape[0]} samples")
print(f"CD vs nonIBD: {X_cd.shape[0]} samples")

UC vs nonIBD: 56 samples
CD vs nonIBD: 75 samples


### 3. Custom Kruskal-Wallis Feature Selector
To prevent data leakage, feature selection must occur *inside* the cross-validation pipeline (on the training fold only). Scikit-learn handles this natively for PCA, but we must build a custom estimator for Kruskal-Wallis. 

*Update:* Per our methodology, we have hardcoded the selection threshold to extract only genera with a **raw $p < 0.01$**.

In [3]:
class KruskalSelector(BaseEstimator, TransformerMixin):
    def __init__(self, p_threshold=0.01):
        self.p_threshold = p_threshold
        self.selected_features_ = None
        
    def fit(self, X, y):
        p_values = []
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        y_arr = y.values if isinstance(y, pd.Series) else y
        
        for i in range(X_arr.shape[1]):
            group0 = X_arr[:, i][y_arr == 0]
            group1 = X_arr[:, i][y_arr == 1]
            stat, p = kruskal(group0, group1)
            p_values.append(p)
            
        p_series = pd.Series(p_values, index=X.columns if isinstance(X, pd.DataFrame) else None)
        
        # Keep features below the strict 0.01 threshold
        self.selected_indices_ = np.where(p_series < self.p_threshold)[0]
        
        # Safety Fallback: If no features meet the threshold in a specific CV fold, grab the top 5 to prevent crashes
        if len(self.selected_indices_) == 0:
            self.selected_indices_ = np.argsort(p_values)[:5]
            
        if isinstance(X, pd.DataFrame):
            self.selected_features_ = X.columns[self.selected_indices_]
            
        return self
        
    def transform(self, X):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        return X_arr[:, self.selected_indices_]

### 4. Hyperparameter Grids & Pipeline Construction
Since our feature selectors are mathematically fixed, we focus the `RandomizedSearchCV` exclusively on tuning the classifier algorithms (SVM and Logistic Regression).

In [4]:
# --- Hyperparameter Grids ---
# We prefix parameters with 'clf__' so the pipeline knows these belong to the Classifier step.

svm_param_grid = {
    'clf__C': np.logspace(-3, 3, 15),
    'clf__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'clf__gamma': ['scale', 'auto', 0.0001, 0.001, 0.01, 0.1, 1],
    'clf__degree': [2, 3],
    'clf__coef0': [0.0, 0.5, 1.0]
}

logreg_param_grid = {
    'clf__C': np.logspace(-4, 4, 20),
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear', 'saga'], # Solvers that support both l1 and l2 penalties
    'clf__max_iter': [2000]
}

# --- Standardized Training Function ---
def build_and_tune_model(X, y, task_name, selector_type, clf_type):
    print(f"\nTraining: {task_name} | Features: {selector_type} | Classifier: {clf_type}")
    
    # 1. Define the Feature Extraction step
    if selector_type == "KW":
        feature_step = ('feat_sel', KruskalSelector(p_threshold=0.01))
    elif selector_type == "PCA":
        # svd_solver='full' allows n_components to be a float representing variance %
        feature_step = ('feat_sel', PCA(n_components=0.60, svd_solver='full', random_state=42))
        
    # 2. Define the Classifier step and select the appropriate grid
    if clf_type == "SVM":
        clf_step = ('clf', SVC(probability=True, random_state=42, class_weight='balanced'))
        param_grid = svm_param_grid
    elif clf_type == "LogReg":
        clf_step = ('clf', LogisticRegression(random_state=42, class_weight='balanced'))
        param_grid = logreg_param_grid

    # 3. Assemble the Pipeline (Scaler goes BEFORE PCA, AFTER KW)
    if selector_type == "PCA":
        pipeline = Pipeline([('scaler', StandardScaler()), feature_step, clf_step])
    else: # For KW, we select first, then scale to save computation
        pipeline = Pipeline([feature_step, ('scaler', StandardScaler()), clf_step])

    # 4. Search and Optimize (50 iterations is plenty since feature selection is fixed)
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_grid, 
        n_iter=50, scoring='roc_auc', 
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
        n_jobs=-1, random_state=42
    )
    
    search.fit(X, y)
    best_model = search.best_estimator_
    
    print(f"  > Best Optimization AUC: {search.best_score_:.3f}")
    
    # Save to disk
    file_name = f"../results/final_{task_name.lower().split()[0]}_{selector_type.lower()}_{clf_type.lower()}.pkl"
    joblib.dump(best_model, file_name)
    
    return best_model, file_name

### 5. Execute Pipeline Construction for All 8 Configurations
We iteratively build, tune, and save the absolute best model for all 8 experimental branches. 
*(Note: Optimization AUC is from a single random seed; true robust performance is calculated in Step 6).*

In [5]:
print("=== Starting Model Generation Phase ===")

saved_models = {}

# UC vs nonIBD
saved_models["UC_KW_SVM"]    = build_and_tune_model(X_uc, y_uc, "UC vs nonIBD", "KW", "SVM")
saved_models["UC_KW_LogReg"] = build_and_tune_model(X_uc, y_uc, "UC vs nonIBD", "KW", "LogReg")
saved_models["UC_PCA_SVM"]   = build_and_tune_model(X_uc, y_uc, "UC vs nonIBD", "PCA", "SVM")
saved_models["UC_PCA_LogReg"]= build_and_tune_model(X_uc, y_uc, "UC vs nonIBD", "PCA", "LogReg")

# CD vs nonIBD
saved_models["CD_KW_SVM"]    = build_and_tune_model(X_cd, y_cd, "CD vs nonIBD", "KW", "SVM")
saved_models["CD_KW_LogReg"] = build_and_tune_model(X_cd, y_cd, "CD vs nonIBD", "KW", "LogReg")
saved_models["CD_PCA_SVM"]   = build_and_tune_model(X_cd, y_cd, "CD vs nonIBD", "PCA", "SVM")
saved_models["CD_PCA_LogReg"]= build_and_tune_model(X_cd, y_cd, "CD vs nonIBD", "PCA", "LogReg")

print("\nAll 8 models tuned and saved to disk.")

=== Starting Model Generation Phase ===

Training: UC vs nonIBD | Features: KW | Classifier: SVM
  > Best Optimization AUC: 0.579

Training: UC vs nonIBD | Features: KW | Classifier: LogReg
  > Best Optimization AUC: 0.578

Training: UC vs nonIBD | Features: PCA | Classifier: SVM
  > Best Optimization AUC: 0.630

Training: UC vs nonIBD | Features: PCA | Classifier: LogReg
  > Best Optimization AUC: 0.558

Training: CD vs nonIBD | Features: KW | Classifier: SVM
  > Best Optimization AUC: 0.649

Training: CD vs nonIBD | Features: KW | Classifier: LogReg
  > Best Optimization AUC: 0.637

Training: CD vs nonIBD | Features: PCA | Classifier: SVM
  > Best Optimization AUC: 0.709

Training: CD vs nonIBD | Features: PCA | Classifier: LogReg
  > Best Optimization AUC: 0.709

All 8 models tuned and saved to disk.


### 6. Robustness Evaluation (The Final Leaderboard)
To combat optimization bias, we load the saved pipelines and subject them to a 3-run $\times$ 5-fold (15 total folds) cross-validation loop. This averages out "lucky" data shuffles and provides the honest, expected metric profile for each model.

In [6]:
def evaluate_multiple_runs(pipeline, X, y, model_name, n_runs=3, n_splits=5):
    """Runs CV across multiple random seeds to get true baseline performance."""
    metrics = {'roc_auc': [], 'accuracy': [], 'f1': [], 'precision': [], 'recall': []}
    
    for run in range(n_runs):
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        scores = cross_validate(pipeline, X, y, cv=cv, 
                                scoring=['roc_auc', 'accuracy', 'f1', 'precision', 'recall'], 
                                n_jobs=-1)
        
        for metric in metrics.keys():
            metrics[metric].extend(scores[f'test_{metric}'])
            
    summary = {'Model Configuration': model_name}
    for metric, values in metrics.items():
        clean_name = metric.replace("test_", "").replace("roc_auc", "AUC").capitalize()
        if clean_name == "Roc_auc": clean_name = "AUC"
        if clean_name == "F1": clean_name = "F1-Score"
        
        summary[clean_name] = f"{np.mean(values):.3f} ± {np.std(values):.3f}"
        
        # Store raw mean AUC for sorting the final table
        if metric == 'roc_auc':
            summary['_sort_auc'] = np.mean(values)
            
    return summary

print("=== Running Robustness Evaluation (15 evaluations per model) ===")

results_list = []

# Evaluate all 8 models
for config_name, (model, file_path) in saved_models.items():
    # Determine the correct dataset
    X_target, y_target = (X_uc, y_uc) if "UC" in config_name else (X_cd, y_cd)
    
    summary = evaluate_multiple_runs(model, X_target, y_target, config_name)
    results_list.append(summary)

# Compile into a final DataFrame
df_final = pd.DataFrame(results_list)

# Split into UC and CD tables for clear reading, sort by AUC, and drop the hidden sort column
df_uc_final = df_final[df_final['Model Configuration'].str.contains("UC")].sort_values('_sort_auc', ascending=False).drop(columns=['_sort_auc'])
df_cd_final = df_final[df_final['Model Configuration'].str.contains("CD")].sort_values('_sort_auc', ascending=False).drop(columns=['_sort_auc'])

print("\n--- ULTIMATE LEADERBOARD: UC vs nonIBD ---")
display(df_uc_final.reset_index(drop=True))

print("\n--- ULTIMATE LEADERBOARD: CD vs nonIBD ---")
display(df_cd_final.reset_index(drop=True))

=== Running Robustness Evaluation (15 evaluations per model) ===

--- ULTIMATE LEADERBOARD: UC vs nonIBD ---


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,UC_PCA_SVM,0.578 ± 0.171,0.547 ± 0.168,0.512 ± 0.236,0.556 ± 0.232,0.500 ± 0.258
1,UC_PCA_LogReg,0.552 ± 0.158,0.549 ± 0.117,0.538 ± 0.167,0.586 ± 0.156,0.533 ± 0.229
2,UC_KW_SVM,0.487 ± 0.143,0.471 ± 0.146,0.457 ± 0.192,0.518 ± 0.222,0.456 ± 0.231
3,UC_KW_LogReg,0.477 ± 0.181,0.490 ± 0.130,0.512 ± 0.146,0.526 ± 0.137,0.522 ± 0.191



--- ULTIMATE LEADERBOARD: CD vs nonIBD ---


,Model Configuration,Auc,Accuracy,F1-Score,Precision,Recall
0,CD_PCA_SVM,0.663 ± 0.145,0.578 ± 0.155,0.624 ± 0.178,0.715 ± 0.166,0.584 ± 0.226
1,CD_PCA_LogReg,0.659 ± 0.147,0.569 ± 0.131,0.613 ± 0.151,0.723 ± 0.148,0.558 ± 0.194
2,CD_KW_SVM,0.606 ± 0.132,0.613 ± 0.117,0.701 ± 0.205,0.639 ± 0.095,0.844 ± 0.314
3,CD_KW_LogReg,0.597 ± 0.131,0.538 ± 0.123,0.576 ± 0.145,0.704 ± 0.162,0.501 ± 0.157
